# CNIBP One-Click (VSCode + Colab)\n在 VSCode 的 Colab 插件连接远程 runtime 后，从上到下 `Run All`。

In [ ]:
# 只需要改这3项
GIT_REPO = 'https://github.com/67vmg9wrfn-beep/Lab.git'
GIT_BRANCH = 'main'
PROJECT_SUBDIR = '06_experiments/cnibp/repro_ppg_bp'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess
if os.path.exists('/content/repo_src'):
    shutil.rmtree('/content/repo_src')
subprocess.run(['git','clone','--depth','1','--branch', GIT_BRANCH, GIT_REPO, '/content/repo_src'], check=True)
print('git clone done')


In [ ]:
import os
PROJECT_ROOT = f'/content/repo_src/{PROJECT_SUBDIR}'
print('PROJECT_ROOT=', PROJECT_ROOT)
assert os.path.exists(PROJECT_ROOT), f'Path not found: {PROJECT_ROOT}'

In [ ]:
from pathlib import Path
import glob

candidate_dirs = [
    '/content/drive/MyDrive/bp_kachuee_cach/raw_mat',
    '/content/drive/MyDrive/bp_kachuee_cach',
    '/content/drive/MyDrive/bp_kachuee_cache',
]

DATA_ROOT = None
for d in candidate_dirs:
    if Path(d).exists():
        parts = [f'{d}/Part_{i}.mat' for i in range(5)]
        if all(Path(x).exists() for x in parts):
            DATA_ROOT = d
            break

if DATA_ROOT is None:
    for d in glob.glob('/content/drive/MyDrive/**', recursive=True):
        if 'kachuee' in d.lower() and Path(d).is_dir():
            parts = [f'{d}/Part_{i}.mat' for i in range(5)]
            if all(Path(x).exists() for x in parts):
                DATA_ROOT = d
                break

if DATA_ROOT is None:
    raise FileNotFoundError('Could not find Part_0.mat..Part_4.mat under /content/drive/MyDrive.')

print('[OK] DATA_ROOT =', DATA_ROOT)


In [ ]:
import subprocess
subprocess.run(['python','-m','pip','install','-r', f'{PROJECT_ROOT}/requirements_colab.txt'], check=True)
subprocess.run(['python','-m','pip','install','-e', PROJECT_ROOT], check=True)
print('dependencies installed')


In [ ]:
import os, subprocess
OUT_ROOT='/content/drive/MyDrive/cnibp_repro_outputs'
os.makedirs(OUT_ROOT, exist_ok=True)
log_file=f'{OUT_ROOT}/last_run.log'
cmd=[
    'python','-m','cnibp_repro.run_repro',
    '--drive_root', DATA_ROOT,
    '--config',f'{PROJECT_ROOT}/configs/paper_repro.json',
    '--output_root',OUT_ROOT
]
with open(log_file, 'w', encoding='utf-8') as f:
    proc=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        f.write(line)
    code=proc.wait()
if code != 0:
    raise RuntimeError(f'run_repro failed, see {log_file}')
print(f'log saved: {log_file}')


In [ ]:
# 失败时你只需要看这个输出路径
print('/content/drive/MyDrive/cnibp_repro_outputs/last_run.log')